#IMPORTS

In [ ]:
from google.colab import drive
from pathlib import Path
import random
import math
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torchvision import models
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

#GLOBALS

In [ ]:
#dataset paths
drive.mount('/content/drive')
src_root = Path("/content/drive/MyDrive/RRDataset_final")
dst_root = Path("/content/drive/MyDrive/RRDataset_split_25")
metadata_path = dst_root / "split_metadata.csv"

#ai/real paths
best_rf_model_path = dst_root / "best_real_fake_resnet18_b128_e50_p10.pth"
history_rf_path = dst_root / "history_real_fake_baseline_b128_e50_p10.csv"

#version paths
best_v_model_path = dst_root / "best_version_resnet18_b128_e50_p10.pth"
history_v_path = dst_root / "history_version_baseline_b128_e50_p10.csv"

#joint paths
best_joint_model_path = dst_root / "best_joint_resnet18_b128_e50_p10_multi_g1_d075.pth"
history_joint_path = dst_root / "history_joint_baseline_b128_e50_p10_multi_g1_d075.csv"

Image.MAX_IMAGE_PIXELS = 1000000000
#dataset globals
FRACTION = 0.25
SEED = 23
preserve_metadata = True
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
}
PATCH_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
PIN_MEMORY = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
PATIENCE = 10
GAMMA = 1       #weight of ai/real on joint loss
DELTA = 0.25        #weight of version on joint loss

#UTILS

##Data utils

In [ ]:
def infer_ai_real_from_filename(path: Path):
    stem = path.stem.lower()
    if "real" in stem:
        return "real"
    else:
        return "ai"

def infer_version_from_filename(path: Path):
    stem = path.stem.lower()
    if "transfer" in stem:
        return "transfer"
    elif "redigital" in stem:
        return "redigital"
    else:
        return "original"

def get_base_identity(path: Path):
    stem = path.stem.lower()
    version_words = {"redigital", "transfer", "original"}
    parts = stem.split("_")
    parts = [p for p in parts if p not in version_words]
    return "_".join(parts)


def make_group_id(path: Path, ai_real: str):
    base_identity = get_base_identity(path)
    if ai_real == "real":
        return base_identity
    else:
        return f"ai_{base_identity}"

def assign_split(group_id):
    if group_id in train_group_ids:
        return "train"
    elif group_id in val_group_ids:
        return "val"
    elif group_id in test_group_ids:
        return "test"
    else:
        raise RuntimeError(f"Group not assigned to any split: {group_id}")

def pad_if_needed(image, min_size=PATCH_SIZE, fill=0, padding_mode="reflect"):
    width, height = image.size
    pad_width = max(0, min_size - width)
    pad_height = max(0, min_size - height)
    if pad_width == 0 and pad_height == 0:
        return image
    left = pad_width // 2
    right = pad_width - left
    top = pad_height // 2
    bottom = pad_height - top
    image = F.pad(
        image,
        padding=[left, top, right, bottom],
        fill=fill,
        padding_mode=padding_mode
    )
    return image

class RRMultiTaskDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.transform = transform
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = self.root_dir / row["reduced_path"]
        if not image_path.exists():
            raise FileNotFoundError(f"Image not found: {image_path}")
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        ai_real_label = torch.tensor(row["ai_real_label"], dtype=torch.long)
        version_label = torch.tensor(row["version_label"], dtype=torch.long)
        return image, ai_real_label, version_label

##Network utils

In [ ]:
def train_one_epoch_real_fake(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    for images, ai_real_labels, version_labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        ai_real_labels = ai_real_labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, ai_real_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(ai_real_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1 = f1_score(all_targets, all_preds, average="binary", zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1

def train_one_epoch_version(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    for images, ai_real_labels, version_labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        version_labels = version_labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, version_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(version_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1


def train_one_epoch_joint(
    model,
    dataloader,
    criterion_rf,
    criterion_v,
    optimizer,
    device,
    gamma=GAMMA,
    delta=DELTA
):
    model.train()
    running_loss = 0.0
    running_rf_loss = 0.0
    running_v_loss = 0.0
    all_rf_preds = []
    all_rf_targets = []
    all_v_preds = []
    all_v_targets = []
    for images, ai_real_labels, version_labels in tqdm(
        dataloader,
        desc="Training joint",
        leave=False
    ):
        images = images.to(device)
        ai_real_labels = ai_real_labels.to(device)
        version_labels = version_labels.to(device)
        optimizer.zero_grad()
        rf_outputs, v_outputs = model(images)
        loss_rf = criterion_rf(rf_outputs, ai_real_labels)
        loss_v = criterion_v(v_outputs, version_labels)
        loss = gamma * loss_rf + delta * loss_v
        loss.backward()
        optimizer.step()
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        running_rf_loss += loss_rf.item() * batch_size
        running_v_loss += loss_v.item() * batch_size
        rf_preds = torch.argmax(rf_outputs, dim=1)
        v_preds = torch.argmax(v_outputs, dim=1)
        all_rf_preds.extend(rf_preds.detach().cpu().numpy())
        all_rf_targets.extend(ai_real_labels.detach().cpu().numpy())
        all_v_preds.extend(v_preds.detach().cpu().numpy())
        all_v_targets.extend(version_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_rf_loss = running_rf_loss / len(dataloader.dataset)
    epoch_v_loss = running_v_loss / len(dataloader.dataset)
    rf_acc = accuracy_score(all_rf_targets, all_rf_preds)
    rf_f1 = f1_score(
        all_rf_targets,
        all_rf_preds,
        average="binary",
        zero_division=0
    )
    v_acc = accuracy_score(all_v_targets, all_v_preds)
    v_f1 = f1_score(
        all_v_targets,
        all_v_preds,
        average="macro",
        zero_division=0
    )
    joint_f1 = (rf_f1 + v_f1) / 2.0
    return {
        "loss": epoch_loss,
        "rf_loss": epoch_rf_loss,
        "version_loss": epoch_v_loss,
        "rf_acc": rf_acc,
        "rf_f1": rf_f1,
        "version_acc": v_acc,
        "version_f1": v_f1,
        "joint_f1": joint_f1
    }


def evaluate_real_fake(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, ai_real_labels, version_labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images = images.to(device)
            ai_real_labels = ai_real_labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, ai_real_labels)
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(ai_real_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_precision = precision_score(all_targets, all_preds, average="binary", zero_division=0)
    epoch_recall = recall_score(all_targets, all_preds, average="binary", zero_division=0)
    epoch_f1 = f1_score(all_targets, all_preds, average="binary", zero_division=0)
    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "precision": epoch_precision,
        "recall": epoch_recall,
        "f1": epoch_f1,
        "targets": all_targets,
        "preds": all_preds
    }


def evaluate_real_fake_multi_patch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, ai_real_labels, version_labels in tqdm(dataloader, desc="Evaluating multi-patch", leave=False):
            images = images.to(device)
            ai_real_labels = ai_real_labels.to(device)
            batch_size, num_patches, C, H, W = images.shape
            images = images.view(batch_size * num_patches, C, H, W)
            outputs = model(images)
            outputs = outputs.view(batch_size, num_patches, -1)
            outputs_mean = outputs.mean(dim=1)                                  #average logits over patches
            loss = criterion(outputs_mean, ai_real_labels)
            running_loss += loss.item() * batch_size
            preds = torch.argmax(outputs_mean, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(ai_real_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_precision = precision_score(all_targets, all_preds, average="binary", zero_division=0)
    epoch_recall = recall_score(all_targets, all_preds, average="binary", zero_division=0)
    epoch_f1 = f1_score(all_targets, all_preds, average="binary", zero_division=0)
    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "precision": epoch_precision,
        "recall": epoch_recall,
        "f1": epoch_f1,
        "targets": all_targets,
        "preds": all_preds
    }

def evaluate_version(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, ai_real_labels, version_labels in tqdm(
            dataloader,
            desc="Evaluating",
            leave=False
        ):
            images = images.to(device)
            version_labels = version_labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, version_labels)
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(version_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_precision = precision_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )
    epoch_recall = recall_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )
    epoch_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )
    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "precision": epoch_precision,
        "recall": epoch_recall,
        "f1": epoch_f1,
        "targets": all_targets,
        "preds": all_preds
    }

def evaluate_version_multi_patch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, ai_real_labels, version_labels in tqdm(dataloader,desc="Evaluating multi-patch",leave=False):
            images = images.to(device)
            version_labels = version_labels.to(device)
            batch_size, num_patches, C, H, W = images.shape
            images = images.view(batch_size * num_patches, C, H, W)
            outputs = model(images)
            outputs = outputs.view(batch_size, num_patches, -1)
            outputs_mean = outputs.mean(dim=1)                                  #average logits over patches
            loss = criterion(outputs_mean, version_labels)
            running_loss += loss.item() * batch_size
            preds = torch.argmax(outputs_mean, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(version_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_precision = precision_score(all_targets, all_preds, average="macro", zero_division=0)
    epoch_recall = recall_score(all_targets, all_preds, average="macro", zero_division=0)
    epoch_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "precision": epoch_precision,
        "recall": epoch_recall,
        "f1": epoch_f1,
        "targets": all_targets,
        "preds": all_preds
    }


def evaluate_joint_multi_patch(
    model,
    dataloader,
    criterion_rf,
    criterion_v,
    device,
    gamma=GAMMA,
    delta=DELTA
):
    model.eval()
    running_loss = 0.0
    running_rf_loss = 0.0
    running_v_loss = 0.0
    all_rf_preds = []
    all_rf_targets = []
    all_v_preds = []
    all_v_targets = []
    with torch.no_grad():
        for images, ai_real_labels, version_labels in tqdm(
            dataloader,
            desc="Evaluating joint multi-patch",
            leave=False
        ):
            images = images.to(device)
            ai_real_labels = ai_real_labels.to(device)
            version_labels = version_labels.to(device)
            batch_size, num_patches, C, H, W = images.shape
            images = images.view(batch_size * num_patches, C, H, W)
            rf_outputs, v_outputs = model(images)
            rf_outputs = rf_outputs.view(batch_size, num_patches, -1)
            v_outputs = v_outputs.view(batch_size, num_patches, -1)
            rf_outputs_mean = rf_outputs.mean(dim=1)                            # average logits over patches
            v_outputs_mean = v_outputs.mean(dim=1)
            loss_rf = criterion_rf(rf_outputs_mean, ai_real_labels)
            loss_v = criterion_v(v_outputs_mean, version_labels)
            loss = gamma * loss_rf + delta * loss_v
            running_loss += loss.item() * batch_size
            running_rf_loss += loss_rf.item() * batch_size
            running_v_loss += loss_v.item() * batch_size
            rf_preds = torch.argmax(rf_outputs_mean, dim=1)
            v_preds = torch.argmax(v_outputs_mean, dim=1)
            all_rf_preds.extend(rf_preds.detach().cpu().numpy())
            all_rf_targets.extend(ai_real_labels.detach().cpu().numpy())
            all_v_preds.extend(v_preds.detach().cpu().numpy())
            all_v_targets.extend(version_labels.detach().cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_rf_loss = running_rf_loss / len(dataloader.dataset)
    epoch_v_loss = running_v_loss / len(dataloader.dataset)
    rf_acc = accuracy_score(all_rf_targets, all_rf_preds)
    rf_precision = precision_score(
        all_rf_targets,
        all_rf_preds,
        average="binary",
        zero_division=0
    )
    rf_recall = recall_score(
        all_rf_targets,
        all_rf_preds,
        average="binary",
        zero_division=0
    )
    rf_f1 = f1_score(
        all_rf_targets,
        all_rf_preds,
        average="binary",
        zero_division=0
    )
    v_acc = accuracy_score(all_v_targets, all_v_preds)
    v_precision = precision_score(
        all_v_targets,
        all_v_preds,
        average="macro",
        zero_division=0
    )
    v_recall = recall_score(
        all_v_targets,
        all_v_preds,
        average="macro",
        zero_division=0
    )
    v_f1 = f1_score(
        all_v_targets,
        all_v_preds,
        average="macro",
        zero_division=0
    )
    joint_f1 = (rf_f1 + v_f1) / 2.0
    return {
        "loss": epoch_loss,
        "rf_loss": epoch_rf_loss,
        "version_loss": epoch_v_loss,
        "rf_accuracy": rf_acc,
        "rf_precision": rf_precision,
        "rf_recall": rf_recall,
        "rf_f1": rf_f1,
        "version_accuracy": v_acc,
        "version_precision": v_precision,
        "version_recall": v_recall,
        "version_f1": v_f1,
        "joint_f1": joint_f1,
        "rf_targets": all_rf_targets,
        "rf_preds": all_rf_preds,
        "version_targets": all_v_targets,
        "version_preds": all_v_preds
    }

def evaluate_joint_multi_patch_with_disagreement(
    model,
    dataloader,
    criterion_rf,
    criterion_v,
    device,
    gamma=GAMMA,
    delta=DELTA
):
    model.eval()
    running_loss = 0.0
    running_rf_loss = 0.0
    running_v_loss = 0.0
    all_rf_preds = []
    all_rf_targets = []
    all_v_preds = []
    all_v_targets = []
    # patch disagreement outputs
    all_rf_patch_agreements = []
    all_rf_patch_disagreements = []
    all_rf_ai_prob_std = []
    all_rf_patch_preds = []
    all_v_patch_agreements = []
    all_v_patch_disagreements = []
    all_v_final_prob_std = []
    all_v_patch_preds = []
    all_joint_patch_agreements = []
    all_joint_patch_disagreements = []
    all_joint_patch_preds = []
    with torch.no_grad():
        for images, ai_real_labels, version_labels in tqdm(
            dataloader,
            desc="Evaluating joint multi-patch + disagreement",
            leave=False
        ):
            images = images.to(device)
            ai_real_labels = ai_real_labels.to(device)
            version_labels = version_labels.to(device)
            batch_size, num_patches, C, H, W = images.shape
            images = images.view(batch_size * num_patches, C, H, W)
            rf_outputs, v_outputs = model(images)
            rf_outputs = rf_outputs.view(batch_size, num_patches, -1)
            v_outputs = v_outputs.view(batch_size, num_patches, -1)
            rf_patch_preds = torch.argmax(rf_outputs, dim=2)
            v_patch_preds = torch.argmax(v_outputs, dim=2)
            rf_patch_probs = torch.softmax(rf_outputs, dim=2)
            v_patch_probs = torch.softmax(v_outputs, dim=2)
            rf_patch_ai_probs = rf_patch_probs[:, :, 1]
            rf_ai_prob_std = rf_patch_ai_probs.std(dim=1, unbiased=False)
            rf_outputs_mean = rf_outputs.mean(dim=1)
            v_outputs_mean = v_outputs.mean(dim=1)
            loss_rf = criterion_rf(rf_outputs_mean, ai_real_labels)
            loss_v = criterion_v(v_outputs_mean, version_labels)
            loss = gamma * loss_rf + delta * loss_v
            running_loss += loss.item() * batch_size
            running_rf_loss += loss_rf.item() * batch_size
            running_v_loss += loss_v.item() * batch_size
            rf_preds = torch.argmax(rf_outputs_mean, dim=1)
            v_preds = torch.argmax(v_outputs_mean, dim=1)
            rf_patch_agreement = (
                rf_patch_preds == rf_preds.unsqueeze(1)
            ).float().mean(dim=1)
            rf_patch_disagreement = 1.0 - rf_patch_agreement
            v_patch_agreement = (
                v_patch_preds == v_preds.unsqueeze(1)
            ).float().mean(dim=1)
            v_patch_disagreement = 1.0 - v_patch_agreement
            v_final_class_probs = v_patch_probs.gather(
                dim=2,
                index=v_preds.view(batch_size, 1, 1).expand(-1, num_patches, 1)
            ).squeeze(2)
            v_final_prob_std = v_final_class_probs.std(dim=1, unbiased=False)
            joint_patch_preds = rf_patch_preds * 3 + v_patch_preds
            joint_preds = rf_preds * 3 + v_preds
            joint_patch_agreement = (
                joint_patch_preds == joint_preds.unsqueeze(1)
            ).float().mean(dim=1)
            joint_patch_disagreement = 1.0 - joint_patch_agreement
            all_rf_preds.extend(rf_preds.detach().cpu().numpy())
            all_rf_targets.extend(ai_real_labels.detach().cpu().numpy())
            all_v_preds.extend(v_preds.detach().cpu().numpy())
            all_v_targets.extend(version_labels.detach().cpu().numpy())
            all_rf_patch_agreements.extend(
                rf_patch_agreement.detach().cpu().numpy()
            )
            all_rf_patch_disagreements.extend(
                rf_patch_disagreement.detach().cpu().numpy()
            )
            all_rf_ai_prob_std.extend(
                rf_ai_prob_std.detach().cpu().numpy()
            )
            all_rf_patch_preds.extend(
                rf_patch_preds.detach().cpu().numpy().tolist()
            )
            all_v_patch_agreements.extend(
                v_patch_agreement.detach().cpu().numpy()
            )
            all_v_patch_disagreements.extend(
                v_patch_disagreement.detach().cpu().numpy()
            )
            all_v_final_prob_std.extend(
                v_final_prob_std.detach().cpu().numpy()
            )
            all_v_patch_preds.extend(
                v_patch_preds.detach().cpu().numpy().tolist()
            )
            all_joint_patch_agreements.extend(
                joint_patch_agreement.detach().cpu().numpy()
            )
            all_joint_patch_disagreements.extend(
                joint_patch_disagreement.detach().cpu().numpy()
            )
            all_joint_patch_preds.extend(
                joint_patch_preds.detach().cpu().numpy().tolist()
            )
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_rf_loss = running_rf_loss / len(dataloader.dataset)
    epoch_v_loss = running_v_loss / len(dataloader.dataset)
    rf_acc = accuracy_score(all_rf_targets, all_rf_preds)
    rf_precision = precision_score(
        all_rf_targets,
        all_rf_preds,
        average="binary",
        zero_division=0
    )
    rf_recall = recall_score(
        all_rf_targets,
        all_rf_preds,
        average="binary",
        zero_division=0
    )
    rf_f1 = f1_score(
        all_rf_targets,
        all_rf_preds,
        average="binary",
        zero_division=0
    )
    v_acc = accuracy_score(all_v_targets, all_v_preds)
    v_precision = precision_score(
        all_v_targets,
        all_v_preds,
        average="macro",
        zero_division=0
    )
    v_recall = recall_score(
        all_v_targets,
        all_v_preds,
        average="macro",
        zero_division=0
    )
    v_f1 = f1_score(
        all_v_targets,
        all_v_preds,
        average="macro",
        zero_division=0
    )
    joint_f1 = (rf_f1 + v_f1) / 2.0
    return {
        "loss": epoch_loss,
        "rf_loss": epoch_rf_loss,
        "version_loss": epoch_v_loss,
        "rf_accuracy": rf_acc,
        "rf_precision": rf_precision,
        "rf_recall": rf_recall,
        "rf_f1": rf_f1,
        "version_accuracy": v_acc,
        "version_precision": v_precision,
        "version_recall": v_recall,
        "version_f1": v_f1,
        "joint_f1": joint_f1,
        "rf_targets": all_rf_targets,
        "rf_preds": all_rf_preds,
        "version_targets": all_v_targets,
        "version_preds": all_v_preds,

        # Real/fake patch consistency
        "rf_patch_agreement": all_rf_patch_agreements,
        "rf_patch_disagreement": all_rf_patch_disagreements,
        "rf_ai_prob_std": all_rf_ai_prob_std,
        "rf_patch_preds": all_rf_patch_preds,

        # Version patch consistency
        "version_patch_agreement": all_v_patch_agreements,
        "version_patch_disagreement": all_v_patch_disagreements,
        "version_final_prob_std": all_v_final_prob_std,
        "version_patch_preds": all_v_patch_preds,

        # Combined 6-class patch consistency
        "joint_patch_agreement": all_joint_patch_agreements,
        "joint_patch_disagreement": all_joint_patch_disagreements,
        "joint_patch_preds": all_joint_patch_preds
    }

#DATA

##RUN ONCE

In [ ]:
#(BUILDS REDUCED DATASET, OVERWRITES IF ALREADY EXISTS)

random.seed(SEED)
#safety checks------------------------------------------------------------------
if not src_root.exists():
    raise FileNotFoundError(f"Source folder not found: {src_root}")
if not src_root.is_dir():
    raise NotADirectoryError(f"Source path is not a folder: {src_root}")

if abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) > 1e-9:
    raise ValueError("train_ratio + val_ratio + test_ratio must be equal to 1.")

if dst_root.exists():
    print(f"Removing existing folder: {dst_root}")
    shutil.rmtree(dst_root)

dst_root.mkdir(parents=True, exist_ok=True)

#data labeling------------------------------------------------------------------
rows = []
for path in src_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        version = infer_version_from_filename(path)
        ai_real = infer_ai_real_from_filename(path)
        group_id = make_group_id(path, ai_real)
        rows.append({
            "src_path": path,
            "relative_path": path.relative_to(src_root),
            "ai_real": ai_real,
            "version": version,
            "group_id": group_id,
            "combined_label": f"{ai_real}_{version}"
        })
df = pd.DataFrame(rows)
if len(df) == 0:
    raise RuntimeError("No images found.")

#original dataset consistency check---------------------------------------------
group_check = (
    df.groupby("group_id")
    .agg(
        ai_real_unique=("ai_real", "nunique"),                                  #check unique occurences of the ai_real label (must be 1)
        versions=("version", lambda x: sorted(set(x))),
        n_images=("src_path", "count")
    )
    .reset_index()
)
problem_groups = group_check[group_check["ai_real_unique"] != 1]                #problematic groups are the ones that span across real and ai
if len(problem_groups) > 0:
    print("\nWarning: some groups contain both AI and real labels.")
    print(problem_groups)
    raise RuntimeError("Invalid grouping detected.")
print("\nGrouping consistency check passed.")

#reduced balanced dataset-------------------------------------------------------
group_df = (
    df.groupby("group_id")
    .agg(
        ai_real=("ai_real", "first"),
        n_images=("src_path", "count")
    )
    .reset_index()
)

n_groups_total = len(group_df)
n_groups_select = max(1, math.ceil(n_groups_total * FRACTION))                  #subset of groups

print("\nTotal groups before reduction:", n_groups_total)
print("Groups to select:", n_groups_select)


selected_groups, _ = train_test_split(                                          #select the subset
    group_df,
    train_size=n_groups_select,
    random_state=SEED,
    stratify=group_df["ai_real"]
)


selected_group_ids = set(selected_groups["group_id"])

df_reduced = df[df["group_id"].isin(selected_group_ids)].reset_index(drop=True)


#TRAIN/VAL/TEST splitting-------------------------------------------------------
reduced_group_df = (
    df_reduced.groupby("group_id")
    .agg(
        ai_real=("ai_real", "first"),
        n_images=("src_path", "count")
    )
    .reset_index()
)

train_groups, temp_groups = train_test_split(                                   #select the train group
    reduced_group_df,
    train_size=TRAIN_RATIO,
    random_state=SEED,
    stratify=reduced_group_df["ai_real"]
)

relative_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_groups, test_groups = train_test_split(                                     #select val and test groups
    temp_groups,
    test_size=relative_test_ratio,
    random_state=SEED,
    stratify=temp_groups["ai_real"]
)

train_group_ids = set(train_groups["group_id"])
val_group_ids = set(val_groups["group_id"])
test_group_ids = set(test_groups["group_id"])

df_reduced["split"] = df_reduced["group_id"].apply(assign_split)

#data leakage check-------------------------------------------------------------
train_set = set(df_reduced[df_reduced["split"] == "train"]["group_id"])
val_set = set(df_reduced[df_reduced["split"] == "val"]["group_id"])
test_set = set(df_reduced[df_reduced["split"] == "test"]["group_id"])

assert train_set.isdisjoint(val_set), "Leakage between train and val!"
assert train_set.isdisjoint(test_set), "Leakage between train and test!"
assert val_set.isdisjoint(test_set), "Leakage between val and test!"

print("\nLeakage check passed: no group appears in more than one split.")

#copy files---------------------------------------------------------------------
total_copied = 0
copied_paths = []
print("\nCopying files...\n")
for idx, row in df_reduced.iterrows():
    src_path = row["src_path"]
    split = row["split"]
    dst_folder = dst_root / split
    dst_folder.mkdir(parents=True, exist_ok=True)
    dst_path = dst_folder / src_path.name
    if preserve_metadata:
        shutil.copy2(src_path, dst_path)
    else:
        shutil.copyfile(src_path, dst_path)
    copied_paths.append(dst_path.relative_to(dst_root))
    total_copied += 1
df_reduced["reduced_path"] = [str(p) for p in copied_paths]

#save labels in a csv
csv_path = dst_root / "split_metadata.csv"
df_reduced_to_save = df_reduced.copy()
df_reduced_to_save["src_path"] = df_reduced_to_save["src_path"].astype(str)
df_reduced_to_save["relative_path"] = df_reduced_to_save["relative_path"].astype(str)
df_reduced_to_save.to_csv(csv_path, index=False)
print(f"\nSaved split metadata to: {csv_path}")


##RUN EVERYTIME

In [ ]:

#load csv into a dataframe
metadata_df = pd.read_csv(metadata_path)

print("Metadata loaded:", metadata_df.shape)
print(metadata_df.head())

#label encoding

ai_real_to_idx = {
    "real": 0,
    "ai": 1
}

version_to_idx = {
    "original": 0,
    "transfer": 1,
    "redigital": 2
}

#inverse mapping
idx_to_ai_real = {v: k for k, v in ai_real_to_idx.items()}
idx_to_version = {v: k for k, v in version_to_idx.items()}

metadata_df["ai_real_label"] = metadata_df["ai_real"].map(ai_real_to_idx)
metadata_df["version_label"] = metadata_df["version"].map(version_to_idx)

if metadata_df["ai_real_label"].isna().any():
    raise RuntimeError("Some ai_real labels were not mapped correctly.")

if metadata_df["version_label"].isna().any():
    raise RuntimeError("Some version labels were not mapped correctly.")

metadata_df["ai_real_label"] = metadata_df["ai_real_label"].astype(int)
metadata_df["version_label"] = metadata_df["version_label"].astype(int)


#image pre_processing

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: pad_if_needed(img, min_size=PATCH_SIZE)),
    transforms.RandomCrop(PATCH_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],                                             #ResNet was trained with images with this normalization
        std=[0.229, 0.224, 0.225]                                               #hence we applied it to our images
    )
])

eval_transform = transforms.Compose([
    transforms.Lambda(lambda img: pad_if_needed(img, min_size=PATCH_SIZE)),
    transforms.CenterCrop(PATCH_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

multi_patch_eval_transform = transforms.Compose([
    transforms.Lambda(lambda img: pad_if_needed(img, min_size=PATCH_SIZE)),
    transforms.FiveCrop(PATCH_SIZE),
    transforms.Lambda(lambda crops: torch.stack([
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )(transforms.ToTensor()(crop))
        for crop in crops
    ]))
])



#dataframe splitting
train_df = metadata_df[metadata_df["split"] == "train"].reset_index(drop=True)
val_df = metadata_df[metadata_df["split"] == "val"].reset_index(drop=True)
test_df = metadata_df[metadata_df["split"] == "test"].reset_index(drop=True)
print("\nSplit sizes:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

if len(train_df) == 0:
    raise RuntimeError("Train split is empty.")

if len(val_df) == 0:
    raise RuntimeError("Validation split is empty.")

if len(test_df) == 0:
    raise RuntimeError("Test split is empty.")

#datasets for dataloaders

train_dataset = RRMultiTaskDataset(
    dataframe=train_df,
    root_dir=dst_root,
    transform=train_transform
)

val_dataset = RRMultiTaskDataset(
    dataframe=val_df,
    root_dir=dst_root,
    transform=eval_transform
)

val_dataset_multi_patch = RRMultiTaskDataset(
    dataframe=val_df,
    root_dir=dst_root,
    transform=multi_patch_eval_transform
)

test_dataset = RRMultiTaskDataset(
    dataframe=test_df,
    root_dir=dst_root,
    transform=eval_transform
)

test_dataset_multi_patch = RRMultiTaskDataset(
    dataframe=test_df,
    root_dir=dst_root,
    transform=multi_patch_eval_transform
)



#dataloaders (to pass images in batches to the model)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader_multi_patch = DataLoader(
    val_dataset_multi_patch,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader_multi_patch = DataLoader(
    test_dataset_multi_patch,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

#NETWORK

##REAL/AI MODEL

In [ ]:

weights = models.ResNet18_Weights.IMAGENET1K_V1
real_fake_model = models.resnet18(weights=weights)
in_features = real_fake_model.fc.in_features
real_fake_model.fc = nn.Linear(in_features, 2)
real_fake_model = real_fake_model.to(DEVICE)
print(real_fake_model.fc)

criterion_rf = nn.CrossEntropyLoss()
optimizer_rf = torch.optim.AdamW(
    real_fake_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
scheduler_rf_loss = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_rf,
    mode="min",
    factor=0.5,
    patience=2
)
scheduler_rf_f1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_rf,
    mode="max",
    factor=0.5,
    patience=2
)

##VERSION MODEL

In [ ]:

weights = models.ResNet18_Weights.IMAGENET1K_V1
version_model = models.resnet18(weights=weights)
in_features = version_model.fc.in_features
version_model.fc = nn.Linear(in_features, 3)
version_model = version_model.to(DEVICE)
print(version_model.fc)

criterion_v = nn.CrossEntropyLoss()
optimizer_v = torch.optim.AdamW(
    version_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
scheduler_v_loss = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v,
    mode="min",
    factor=0.5,
    patience=2
)
scheduler_v_f1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v,
    mode="max",
    factor=0.5,
    patience=2
)


##JOINT MODEL

In [ ]:

def build_joint_model(num_rf_classes=2, num_version_classes=3, pretrained=True):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Identity()
    model.forward_backbone = model.forward
    model.real_fake_head = nn.Linear(in_features, num_rf_classes)
    model.version_head = nn.Linear(in_features, num_version_classes)

    # Custom forward function
    def forward_joint(x):
        features = model.forward_backbone(x)
        real_fake_logits = model.real_fake_head(features)
        version_logits = model.version_head(features)
        return real_fake_logits, version_logits
    model.forward = forward_joint
    return model


joint_model = build_joint_model(
    num_rf_classes=2,
    num_version_classes=3,
    pretrained=True
)
joint_model = joint_model.to(DEVICE)
print("Joint model heads:")
print("Real/fake head:", joint_model.real_fake_head)
print("Version head:", joint_model.version_head)



criterion_joint_rf = nn.CrossEntropyLoss()
criterion_joint_v = nn.CrossEntropyLoss()

optimizer_joint = torch.optim.AdamW(
    joint_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
scheduler_joint_loss = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_joint,
    mode="min",
    factor=0.5,
    patience=2
)

#TRAIN

##TRAINING REAL/AI

In [ ]:

# REAL/FAKE BASELINE

best_val_f1 = -1.0
patience_counter = 0

history_rf = []

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{NUM_EPOCHS}]")
    train_loss, train_acc, train_f1 = train_one_epoch_real_fake(
        model=real_fake_model,
        dataloader=train_loader,
        criterion=criterion_rf,
        optimizer=optimizer_rf,
        device=DEVICE
    )
    val_results = evaluate_real_fake_multi_patch(
        model=real_fake_model,
        dataloader=val_loader_multi_patch,
        criterion=criterion_rf,
        device=DEVICE
    )
    val_loss = val_results["loss"]
    val_acc = val_results["accuracy"]
    val_precision = val_results["precision"]
    val_recall = val_results["recall"]
    val_f1 = val_results["f1"]
    scheduler_rf_loss.step(val_loss)
    current_lr = optimizer_rf.param_groups[0]["lr"]
    history_rf.append({
        "epoch": epoch + 1,
        "lr": current_lr,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_f1": val_f1
    })

    print(f"Current LR: {current_lr:.2e}")
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   loss: {val_loss:.4f} | Val   acc: {val_acc:.4f} | Val   F1: {val_f1:.4f}")
    print(f"Val precision: {val_precision:.4f} | Val recall: {val_recall:.4f}")

    if val_f1 > best_val_f1:
        patience_counter = 0
        best_val_f1 = val_f1
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": real_fake_model.state_dict(),
            "optimizer_state_dict": optimizer_rf.state_dict(),
            "scheduler_state_dict": scheduler_rf_loss.state_dict(),
            "best_val_f1": best_val_f1,
            "ai_real_to_idx": ai_real_to_idx,
            "idx_to_ai_real": idx_to_ai_real
        }, best_rf_model_path)

        print(f"Saved best model with Val F1 = {best_val_f1:.4f}")
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break
# save training history
history_rf_df = pd.DataFrame(history_rf)

history_rf_df.to_csv(history_rf_path, index=False)
print("Saved training history to:", history_rf_path)
print(history_rf_df)

##TRAINING VERSION

In [ ]:

# VERSION BASELINE

best_val_f1 = -1.0
patience_counter = 0


history_v = []
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{NUM_EPOCHS}]")
    train_loss, train_acc, train_f1 = train_one_epoch_version(
        model=version_model,
        dataloader=train_loader,
        criterion=criterion_v,
        optimizer=optimizer_v,
        device=DEVICE
    )
    val_results = evaluate_version(
        model=version_model,
        dataloader=val_loader,
        criterion=criterion_v,
        device=DEVICE
    )
    val_loss = val_results["loss"]
    val_acc = val_results["accuracy"]
    val_precision = val_results["precision"]
    val_recall = val_results["recall"]
    val_f1 = val_results["f1"]
    scheduler_v_loss.step(val_loss)
    current_lr = optimizer_v.param_groups[0]["lr"]
    history_v.append({
        "epoch": epoch + 1,
        "lr": current_lr,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_f1": val_f1
    })

    print(f"Current LR: {current_lr:.2e}")
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   loss: {val_loss:.4f} | Val   acc: {val_acc:.4f} | Val   F1: {val_f1:.4f}")
    print(f"Val precision: {val_precision:.4f} | Val recall: {val_recall:.4f}")

    if val_f1 > best_val_f1:
        patience_counter = 0
        best_val_f1 = val_f1
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": version_model.state_dict(),
            "optimizer_state_dict": optimizer_v.state_dict(),
            "scheduler_state_dict": scheduler_v_loss.state_dict(),
            "best_val_f1": best_val_f1,
            "version_to_idx": version_to_idx,
            "idx_to_version": idx_to_version
        }, best_v_model_path)

        print(f"Saved best model with Val F1 = {best_val_f1:.4f}")
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

# save training history
history_v_df = pd.DataFrame(history_v)

history_v_df.to_csv(history_v_path, index=False)
print("Saved training history to:", history_v_path)
print(history_v_df)

##TRAINING JOINT

In [ ]:

#  JOINT

best_val_joint_f1 = -1.0
patience_counter = 0

history_joint = []

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{NUM_EPOCHS}]")
    train_results = train_one_epoch_joint(
        model=joint_model,
        dataloader=train_loader,
        criterion_rf=criterion_joint_rf,
        criterion_v=criterion_joint_v,
        optimizer=optimizer_joint,
        device=DEVICE,
        gamma=GAMMA,
        delta=DELTA
    )
    val_results = evaluate_joint_multi_patch(
        model=joint_model,
        dataloader=val_loader_multi_patch,
        criterion_rf=criterion_joint_rf,
        criterion_v=criterion_joint_v,
        device=DEVICE,
        gamma=GAMMA,
        delta=DELTA
    )
    """
    val_results = evaluate_joint_single_patch(
        model=joint_model,
        dataloader=val_loader,
        criterion_rf=criterion_joint_rf,
        criterion_v=criterion_joint_v,
        device=DEVICE,
        gamma=GAMMA,
        delta=DELTA
    )
    """

    val_loss = val_results["loss"]
    val_joint_f1 = val_results["joint_f1"]
    scheduler_joint_loss.step(val_loss)
    current_lr = optimizer_joint.param_groups[0]["lr"]
    history_joint.append({
        "epoch": epoch + 1,
        "lr": current_lr,
        "train_loss": train_results["loss"],
        "train_rf_loss": train_results["rf_loss"],
        "train_version_loss": train_results["version_loss"],
        "train_rf_acc": train_results["rf_acc"],
        "train_rf_f1": train_results["rf_f1"],
        "train_version_acc": train_results["version_acc"],
        "train_version_f1": train_results["version_f1"],
        "train_joint_f1": train_results["joint_f1"],
        "val_loss": val_results["loss"],
        "val_rf_loss": val_results["rf_loss"],
        "val_version_loss": val_results["version_loss"],
        "val_rf_acc": val_results["rf_accuracy"],
        "val_rf_precision": val_results["rf_precision"],
        "val_rf_recall": val_results["rf_recall"],
        "val_rf_f1": val_results["rf_f1"],
        "val_version_acc": val_results["version_accuracy"],
        "val_version_precision": val_results["version_precision"],
        "val_version_recall": val_results["version_recall"],
        "val_version_f1": val_results["version_f1"],
        "val_joint_f1": val_results["joint_f1"],
        "gamma": GAMMA,
        "delta": DELTA
    })

    print(f"Current LR: {current_lr:.2e}")

    print(
        f"Train loss: {train_results['loss']:.4f} | "
        f"RF F1: {train_results['rf_f1']:.4f} | "
        f"Version F1: {train_results['version_f1']:.4f} | "
        f"Joint F1: {train_results['joint_f1']:.4f}"
    )
    print(
        f"Val   loss: {val_results['loss']:.4f} | "
        f"RF F1: {val_results['rf_f1']:.4f} | "
        f"Version F1: {val_results['version_f1']:.4f} | "
        f"Joint F1: {val_results['joint_f1']:.4f}"
    )
    print(
        f"Val RF acc: {val_results['rf_accuracy']:.4f} | "
        f"Val Version acc: {val_results['version_accuracy']:.4f}"
    )

    if val_joint_f1 > best_val_joint_f1:
        patience_counter = 0
        best_val_joint_f1 = val_joint_f1

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": joint_model.state_dict(),
            "optimizer_state_dict": optimizer_joint.state_dict(),
            "scheduler_state_dict": scheduler_joint_loss.state_dict(),
            "best_val_joint_f1": best_val_joint_f1,
            "ai_real_to_idx": ai_real_to_idx,
            "idx_to_ai_real": idx_to_ai_real,
            "version_to_idx": version_to_idx,
            "idx_to_version": idx_to_version,
            "gamma": GAMMA,
            "delta": DELTA
        }, best_joint_model_path)

        print(f"Saved best joint model with Val Joint F1 = {best_val_joint_f1:.4f}")

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break
#save training history

history_joint_df = pd.DataFrame(history_joint)
history_joint_df.to_csv(history_joint_path, index=False)

print("Saved joint training history to:", history_joint_path)
print(history_joint_df)

#EVALUATION

##REAL/AI EVALUATION

In [ ]:

# REAL/AI TESTING


checkpoint = torch.load(best_rf_model_path, map_location=DEVICE)
real_fake_model.load_state_dict(checkpoint["model_state_dict"])
real_fake_model = real_fake_model.to(DEVICE)
test_results_rf = evaluate_real_fake(
    model=real_fake_model,
    dataloader=test_loader,
    criterion=criterion_rf,
    device=DEVICE
)

print("\nREAL/FAKE BASELINE - TEST RESULTS")
print(f"Test loss:      {test_results_rf['loss']:.4f}")
print(f"Test accuracy:  {test_results_rf['accuracy']:.4f}")
print(f"Test precision: {test_results_rf['precision']:.4f}")
print(f"Test recall:    {test_results_rf['recall']:.4f}")
print(f"Test F1-score:  {test_results_rf['f1']:.4f}")

print("\nClassification report:")
print(classification_report(
    test_results_rf["targets"],
    test_results_rf["preds"],
    target_names=["real", "ai"],
    zero_division=0,
    digits = 4
))

print("\nConfusion matrix:")
cm_rf = confusion_matrix(test_results_rf["targets"], test_results_rf["preds"])
print(cm_rf)

##VERSION EVALUATION

In [ ]:

# VERSION TESTING


checkpoint = torch.load(best_v_model_path, map_location=DEVICE)
version_model.load_state_dict(checkpoint["model_state_dict"])
version_model = version_model.to(DEVICE)
test_results_v = evaluate_version(
    model=version_model,
    dataloader=test_loader,
    criterion=criterion_v,
    device=DEVICE
)

print("\nVERSION BASELINE - TEST RESULTS")
print(f"Test loss:      {test_results_v['loss']:.4f}")
print(f"Test accuracy:  {test_results_v['accuracy']:.4f}")
print(f"Test precision: {test_results_v['precision']:.4f}")
print(f"Test recall:    {test_results_v['recall']:.4f}")
print(f"Test F1-score:  {test_results_v['f1']:.4f}")

print("\nClassification report:")
print(classification_report(
    test_results_v["targets"],
    test_results_v["preds"],
    target_names=["original","transfer","redigital"],
    zero_division=0,
    digits = 4
))

print("\nConfusion matrix:")
cm_v = confusion_matrix(test_results_v["targets"], test_results_v["preds"])

print(cm_v)

##JOINT EVALUATION

In [ ]:

# JOINT TESTING


checkpoint = torch.load(best_joint_model_path, map_location=DEVICE)

joint_model.load_state_dict(checkpoint["model_state_dict"])
joint_model = joint_model.to(DEVICE)

test_results_joint = evaluate_joint_multi_patch(
    model=joint_model,
    dataloader=test_loader_multi_patch,
    criterion_rf=criterion_joint_rf,
    criterion_v=criterion_joint_v,
    device=DEVICE,
    gamma=GAMMA,
    delta=DELTA
)

print("\nJOINT MULTI-TASK MODEL - TEST RESULTS")
print(f"Test total loss:       {test_results_joint['loss']:.4f}")
print(f"Test RF loss:          {test_results_joint['rf_loss']:.4f}")
print(f"Test version loss:     {test_results_joint['version_loss']:.4f}")

print("\nREAL/FAKE HEAD - TEST RESULTS")
print(f"Test accuracy:         {test_results_joint['rf_accuracy']:.4f}")
print(f"Test precision:        {test_results_joint['rf_precision']:.4f}")
print(f"Test recall:           {test_results_joint['rf_recall']:.4f}")
print(f"Test F1-score:         {test_results_joint['rf_f1']:.4f}")

print("\nVERSION HEAD - TEST RESULTS")
print(f"Test accuracy:         {test_results_joint['version_accuracy']:.4f}")
print(f"Test precision:        {test_results_joint['version_precision']:.4f}")
print(f"Test recall:           {test_results_joint['version_recall']:.4f}")
print(f"Test F1-score:         {test_results_joint['version_f1']:.4f}")

print("\nJoint F1:")
print(f"{test_results_joint['joint_f1']:.4f}")

print("\nREAL/FAKE classification report:")
print(classification_report(
    test_results_joint["rf_targets"],
    test_results_joint["rf_preds"],
    target_names=["real", "ai"],
    zero_division=0,
    digits = 4

))

print("\nREAL/FAKE confusion matrix:")
cm_joint_rf = confusion_matrix(
    test_results_joint["rf_targets"],
    test_results_joint["rf_preds"]
)
print(cm_joint_rf)

print("\nVERSION classification report:")
print(classification_report(
    test_results_joint["version_targets"],
    test_results_joint["version_preds"],
    target_names=["original", "transfer", "redigital"],
    zero_division=0,
    digits = 4
))

print("\nVERSION confusion matrix:")
cm_joint_v = confusion_matrix(
    test_results_joint["version_targets"],
    test_results_joint["version_preds"]
)
print(cm_joint_v)

joint_class_names = [
    "real_original",
    "real_transfer",
    "real_redigital",
    "ai_original",
    "ai_transfer",
    "ai_redigital"
]

joint_targets = [
    rf_true * 3 + version_true
    for rf_true, version_true in zip(
        test_results_joint["rf_targets"],
        test_results_joint["version_targets"]
    )
]

joint_preds = [
    rf_pred * 3 + version_pred
    for rf_pred, version_pred in zip(
        test_results_joint["rf_preds"],
        test_results_joint["version_preds"]
    )
]

cm_joint_6 = confusion_matrix(
    joint_targets,
    joint_preds,
    labels=list(range(6))
)

cm_joint_6_df = pd.DataFrame(
    cm_joint_6,
    index=[f"true_{name}" for name in joint_class_names],
    columns=[f"pred_{name}" for name in joint_class_names]
)

print("\nJOINT 6-CLASS CONFUSION MATRIX")
print("Rows = true classes")
print("Columns = predicted classes")
print(cm_joint_6_df)

print("\nJOINT 6-CLASS CLASSIFICATION REPORT")
print(classification_report(
    joint_targets,
    joint_preds,
    labels=list(range(6)),
    target_names=joint_class_names,
    zero_division=0,
    digits = 4
))

#disagreement testing

test_results_joint_patch = evaluate_joint_multi_patch_with_disagreement(
    model=joint_model,
    dataloader=test_loader_multi_patch,
    criterion_rf=criterion_joint_rf,
    criterion_v=criterion_joint_v,
    device=DEVICE,
    gamma=GAMMA,
    delta=DELTA
)

joint_patch_df = pd.DataFrame({
    "rf_target": test_results_joint_patch["rf_targets"],
    "rf_pred": test_results_joint_patch["rf_preds"],

    "version_target": test_results_joint_patch["version_targets"],
    "version_pred": test_results_joint_patch["version_preds"],

    "rf_patch_agreement": test_results_joint_patch["rf_patch_agreement"],
    "rf_patch_disagreement": test_results_joint_patch["rf_patch_disagreement"],
    "rf_ai_prob_std": test_results_joint_patch["rf_ai_prob_std"],

    "version_patch_agreement": test_results_joint_patch["version_patch_agreement"],
    "version_patch_disagreement": test_results_joint_patch["version_patch_disagreement"],
    "version_final_prob_std": test_results_joint_patch["version_final_prob_std"],

    "joint_patch_agreement": test_results_joint_patch["joint_patch_agreement"],
    "joint_patch_disagreement": test_results_joint_patch["joint_patch_disagreement"]
})

joint_patch_df["rf_correct"] = (
    joint_patch_df["rf_target"] == joint_patch_df["rf_pred"]
)

joint_patch_df["version_correct"] = (
    joint_patch_df["version_target"] == joint_patch_df["version_pred"]
)

joint_patch_df["joint_correct"] = (
    joint_patch_df["rf_correct"] & joint_patch_df["version_correct"]
)

joint_patch_df["rf_target_name"] = joint_patch_df["rf_target"].map(idx_to_ai_real)
joint_patch_df["rf_pred_name"] = joint_patch_df["rf_pred"].map(idx_to_ai_real)

joint_patch_df["version_target_name"] = joint_patch_df["version_target"].map(idx_to_version)
joint_patch_df["version_pred_name"] = joint_patch_df["version_pred"].map(idx_to_version)

print("\nPATCH DISAGREEMENT ANALYSIS - JOINT MODEL")

print("\nMean patch agreement:")
print(joint_patch_df[
    [
        "rf_patch_agreement",
        "version_patch_agreement",
        "joint_patch_agreement"
    ]
].mean())

print("\nMean patch disagreement:")
print(joint_patch_df[
    [
        "rf_patch_disagreement",
        "version_patch_disagreement",
        "joint_patch_disagreement"
    ]
].mean())

print("\nREAL/FAKE head: patch agreement for correct vs wrong predictions")
print(
joint_patch_df.groupby("rf_correct")[
        ["rf_patch_agreement", "rf_patch_disagreement", "rf_ai_prob_std"]
    ].mean()
)
print("\nVERSION head: patch agreement for correct vs wrong predictions")
print(joint_patch_df.groupby("version_correct")[
        ["version_patch_agreement","version_patch_disagreement","version_final_prob_std"
        ]
    ].mean()
)

print("\nJOINT prediction: patch agreement for correct vs wrong predictions")
print(joint_patch_df.groupby("joint_correct")[
        ["joint_patch_agreement", "joint_patch_disagreement"]
        ].mean()
)

print("\nREAL/FAKE patch agreement by true class")
print(joint_patch_df.groupby("rf_target_name")[
        ["rf_patch_agreement", "rf_patch_disagreement", "rf_ai_prob_std"]
    ].mean()
)
print("\nPatch agreement by true version")
print(joint_patch_df.groupby("version_target_name")[
        ["rf_patch_agreement","version_patch_agreement","joint_patch_agreement"]
    ].mean()
)
print("\nJoint accuracy by joint patch agreement level")
print(joint_patch_df.groupby("joint_patch_agreement")["joint_correct"].mean())
print("\nNumber of samples by joint patch agreement level")
print(joint_patch_df["joint_patch_agreement"].value_counts().sort_index())